# Pipeline de procesamiento de videos propios

Replica el pipeline del paper WLASL para nuestros 20 glosses:
1. Construye `glosses_nuestras.json` con el mismo esquema que `glosses_valid.json`
2. Extrae keypoints con **MediaPipe Holistic** en formato OpenPose (compatible con el código del paper)

**Formato de keypoints (idéntico a `pose_per_individual_videos/`):**
- `pose_keypoints_2d`: 25 joints × [x, y, conf] = 75 valores (BODY_25)
- `hand_left_keypoints_2d` / `hand_right_keypoints_2d`: 21 joints × [x, y, conf] = 63 valores

**Estructura esperada de entrada:**
```
start_kit/videos_nuestros/<gloss>/<video>.mp4
```

In [ ]:
import os
import json
import random
import cv2
import numpy as np
import mediapipe as mp
from pathlib import Path

In [ ]:
# ── Rutas ──────────────────────────────────────────────────────────────────
BASE         = Path("..").resolve()
VIDEOS_DIR   = BASE / "start_kit" / "videos_nuestros"
OUTPUT_JSON  = BASE / "data" / "glosses_nuestras.json"
OUTPUT_POSES = BASE / "data" / "poses_nuestras"

OUTPUT_POSES.mkdir(parents=True, exist_ok=True)

# ── Las 20 glosas del trabajo ───────────────────────────────────────────────
GLOSSES = [
    "computer", "drink", "before", "hot", "like",
    "who", "deaf", "yes", "mother", "hearing",
    "white", "wrong", "candy", "no", "thin",
    "finish", "now", "orange", "study", "bird"
]

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

## Paso 1 — Construir `glosses_nuestras.json`

Para cada glosa, escanea los `.mp4` disponibles y les asigna:
- `video_id` secuencial (`N00001`, `N00002`, ...)
- `split` con ratio 4:1:1 (train:val:test) replicando el paper
- Metadata del video (fps, frames, resolución)
- `bbox` = frame completo (el paper usaba YOLOv3; nosotros usamos el frame entero)

In [ ]:
def get_video_metadata(video_path: Path) -> dict:
    cap    = cv2.VideoCapture(str(video_path))
    fps    = cap.get(cv2.CAP_PROP_FPS)
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    return {"fps": fps, "width": width, "height": height}


def assign_splits(n: int) -> list:
    """
    Ratio 4:1:1 (train:val:test). Garantiza al menos 1 muestra en
    val y test cuando n >= 3, igual que el paper.
    """
    if n == 1:
        return ["train"]
    if n == 2:
        return ["train", "val"]
    n_val   = max(1, round(n / 6))
    n_test  = max(1, round(n / 6))
    n_train = n - n_val - n_test
    splits  = ["train"] * n_train + ["val"] * n_val + ["test"] * n_test
    rng = random.Random(RANDOM_SEED)
    rng.shuffle(splits)
    return splits


def build_glosses_json() -> list:
    dataset       = []
    video_counter = 1

    for gloss in GLOSSES:
        gloss_dir = VIDEOS_DIR / gloss
        if not gloss_dir.exists():
            print(f"  [SKIP] {gloss}: carpeta no encontrada")
            continue

        videos = sorted(gloss_dir.glob("*.mp4"))
        if not videos:
            print(f"  [SKIP] {gloss}: sin archivos .mp4")
            continue

        splits    = assign_splits(len(videos))
        instances = []

        for idx, (video_path, split) in enumerate(zip(videos, splits)):
            video_id = f"N{video_counter:05d}"
            video_counter += 1
            meta = get_video_metadata(video_path)

            # url guarda la ruta relativa al BASE para poder encontrar el archivo
            relative_url = str(video_path.relative_to(BASE))

            instances.append({
                "video_id":     video_id,
                "instance_id":  idx,
                "signer_id":    idx,
                "source":       "nuestros",
                "url":          relative_url,
                "split":        split,
                "variation_id": 0,
                "fps":          meta["fps"],
                "frame_start":  1,
                "frame_end":    -1,
                "bbox":         [0, 0, meta["width"], meta["height"]],
            })

        dataset.append({"gloss": gloss, "instances": instances})
        print(f"  {gloss:12s}: {len(instances)} videos — "
              f"train={splits.count('train')}, val={splits.count('val')}, test={splits.count('test')}")

    return dataset


print("Construyendo glosses_nuestras.json...")
dataset = build_glosses_json()

with open(OUTPUT_JSON, "w") as f:
    json.dump(dataset, f, indent=2)

total = sum(len(e["instances"]) for e in dataset)
print(f"\nGuardado: {OUTPUT_JSON}")
print(f"Total: {len(dataset)} glosas, {total} videos")

## Paso 2 — Extracción de keypoints

### Mapping MediaPipe → OpenPose BODY_25

MediaPipe Holistic devuelve 33 pose landmarks. Los mapeamos a los 25 de BODY_25
para compatibilidad total con el código del paper.

| OpenPose | Joint | MediaPipe |
|:---:|:---|:---|
| 0 | Nose | 0 |
| 1 | Neck | mid(11,12) — hombros |
| 2 | RShoulder | 12 |
| 3 | RElbow | 14 |
| 4 | RWrist | 16 |
| 5 | LShoulder | 11 |
| 6 | LElbow | 13 |
| 7 | LWrist | 15 |
| 8 | MidHip | mid(23,24) — caderas |
| 9-14 | caderas, rodillas, tobillos | 24,26,28,23,25,27 |
| 15-18 | ojos y orejas | 5,2,8,7 |
| 19-24 | pies | 31,—,29,32,—,30 |

Las **manos** (21 landmarks cada una) son idénticas en cantidad y orden entre MediaPipe y OpenPose.

In [ ]:
# Mapping BODY_25 → MediaPipe
# int   → índice directo en MP
# tuple → promedio de dos índices de MP
# None  → sin equivalente (output = [0, 0, 0])
BODY25_TO_MP = [
    0,        # 0  Nose
    (11, 12), # 1  Neck
    12,       # 2  RShoulder
    14,       # 3  RElbow
    16,       # 4  RWrist
    11,       # 5  LShoulder
    13,       # 6  LElbow
    15,       # 7  LWrist
    (23, 24), # 8  MidHip
    24,       # 9  RHip
    26,       # 10 RKnee
    28,       # 11 RAnkle
    23,       # 12 LHip
    25,       # 13 LKnee
    27,       # 14 LAnkle
    5,        # 15 REye
    2,        # 16 LEye
    8,        # 17 REar
    7,        # 18 LEar
    31,       # 19 LBigToe
    None,     # 20 LSmallToe (sin equivalente)
    29,       # 21 LHeel
    32,       # 22 RBigToe
    None,     # 23 RSmallToe (sin equivalente)
    30,       # 24 RHeel
]


def mp_pose_to_openpose(landmarks, width: int, height: int) -> list:
    """Convierte 33 MP landmarks → 75 valores en formato BODY_25 [x,y,conf,...]"""
    lm     = landmarks.landmark
    result = []
    for ref in BODY25_TO_MP:
        if ref is None:
            result.extend([0.0, 0.0, 0.0])
        elif isinstance(ref, tuple):
            a, b = ref
            result.extend([
                (lm[a].x + lm[b].x) / 2 * width,
                (lm[a].y + lm[b].y) / 2 * height,
                (lm[a].visibility + lm[b].visibility) / 2,
            ])
        else:
            result.extend([
                lm[ref].x * width,
                lm[ref].y * height,
                lm[ref].visibility,
            ])
    return result


def mp_hand_to_openpose(landmarks, width: int, height: int) -> list:
    """Convierte 21 MP hand landmarks → 63 valores [x,y,1.0,...]"""
    result = []
    for lm in landmarks.landmark:
        result.extend([lm.x * width, lm.y * height, 1.0])
    return result


def make_keypoints_dict(pose_kp, hand_left_kp, hand_right_kp) -> dict:
    """Arma el dict en el mismo formato que pose_per_individual_videos/."""
    return {
        "version": 1.3,
        "people": [{
            "person_id":               [-1],
            "pose_keypoints_2d":       pose_kp,
            "face_keypoints_2d":       [],
            "hand_left_keypoints_2d":  hand_left_kp,
            "hand_right_keypoints_2d": hand_right_kp,
            "pose_keypoints_3d":       [],
            "face_keypoints_3d":       [],
            "hand_left_keypoints_3d":  [],
            "hand_right_keypoints_3d": [],
        }],
    }


def extract_keypoints_from_video(video_path: Path, output_dir: Path) -> int:
    """
    Procesa un video frame a frame con MediaPipe Holistic.
    Guarda image_XXXXX_keypoints.json por frame en output_dir.
    Devuelve el número de frames procesados.
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    cap    = cv2.VideoCapture(str(video_path))
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    holistic = mp.solutions.holistic.Holistic(
        static_image_mode=False,
        model_complexity=1,
        enable_segmentation=False,
        refine_face_landmarks=False,
    )

    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame_idx += 1
        rgb     = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = holistic.process(rgb)

        pose_kp       = (mp_pose_to_openpose(results.pose_landmarks, width, height)
                         if results.pose_landmarks else [0.0] * 75)
        hand_left_kp  = (mp_hand_to_openpose(results.left_hand_landmarks, width, height)
                         if results.left_hand_landmarks else [0.0] * 63)
        hand_right_kp = (mp_hand_to_openpose(results.right_hand_landmarks, width, height)
                         if results.right_hand_landmarks else [0.0] * 63)

        kp_dict  = make_keypoints_dict(pose_kp, hand_left_kp, hand_right_kp)
        out_file = output_dir / f"image_{frame_idx:05d}_keypoints.json"
        with open(out_file, "w") as f:
            json.dump(kp_dict, f)

    cap.release()
    holistic.close()
    return frame_idx

In [ ]:
def run_keypoint_extraction():
    with open(OUTPUT_JSON) as f:
        dataset = json.load(f)

    all_instances = [(e["gloss"], inst) for e in dataset for inst in e["instances"]]
    total = len(all_instances)

    for i, (gloss, inst) in enumerate(all_instances, 1):
        video_id   = inst["video_id"]
        video_path = BASE / inst["url"]
        output_dir = OUTPUT_POSES / video_id

        if not video_path.exists():
            print(f"  [SKIP] {video_id} ({gloss}): archivo no encontrado")
            continue

        # Saltear si ya está completamente procesado
        cap = cv2.VideoCapture(str(video_path))
        expected = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        cap.release()
        existing = len(list(output_dir.glob("image_*_keypoints.json"))) if output_dir.exists() else 0
        if existing == expected > 0:
            print(f"  [OK]   {video_id} ({gloss}): ya procesado ({expected} frames)")
            continue

        n_frames = extract_keypoints_from_video(video_path, output_dir)
        print(f"  [{i}/{total}] {video_id} ({gloss}): {n_frames} frames")

    print(f"\nListo. Poses en: {OUTPUT_POSES}")


print("Iniciando extracción de keypoints...")
run_keypoint_extraction()

## Verificación de compatibilidad

In [ ]:
def verify_output():
    # glosses_nuestras.json
    with open(OUTPUT_JSON) as f:
        data = json.load(f)
    print("=== glosses_nuestras.json ===")
    print(f"  Glosas: {len(data)}")
    for entry in data:
        insts  = entry["instances"]
        splits = [inst["split"] for inst in insts]
        print(f"  {entry['gloss']:12s}: {len(insts):2d} videos "
              f"(train={splits.count('train')}, val={splits.count('val')}, test={splits.count('test')})")

    # Verificar formato de un JSON de poses
    pose_dirs = sorted([d for d in OUTPUT_POSES.iterdir() if d.is_dir()])
    if not pose_dirs:
        print("\nNo hay poses generadas todavía.")
        return

    sample_file = sorted(pose_dirs[0].glob("image_*_keypoints.json"))[0]
    with open(sample_file) as f:
        kp = json.load(f)

    person  = kp["people"][0]
    n_pose  = len(person["pose_keypoints_2d"]) // 3
    n_left  = len(person["hand_left_keypoints_2d"]) // 3
    n_right = len(person["hand_right_keypoints_2d"]) // 3

    print(f"\n=== Formato de poses (muestra: {sample_file.name}) ===")
    print(f"  pose:       {n_pose:2d} joints  (esperado 25)  {'OK' if n_pose==25 else 'ERROR'}")
    print(f"  hand_left:  {n_left:2d} joints  (esperado 21)  {'OK' if n_left==21 else 'ERROR'}")
    print(f"  hand_right: {n_right:2d} joints  (esperado 21)  {'OK' if n_right==21 else 'ERROR'}")


verify_output()